# 05 · Three-Group Alternative Analysis
### ⚠️ Exploratory robustness check — needs independent confirmation

**Purpose:** this notebook is deliberately **independent** of the main pipeline's chosen archetype solution (notebooks 02-04). It re-runs the embedding and clustering steps from scratch, with a different random seed and a different UMAP neighbourhood size, and **forces a three-group (k=3) solution**, to ask a narrower question:

> *If the lifestyle data are pushed toward a coarser, three-way split, does the direction of the archetype-PIU association still hold — or was the original finding an artefact of one specific parameterisation?*

**This is a robustness probe, not a replacement analysis.** It should be read alongside these caveats:

- The three-group solution here is **not validated to the same standard** as the main pipeline's chosen method (notebook 03) — no dedicated stability/subsampling analysis has been run for it specifically.
- A single alternative run, even if directionally consistent with the main finding, is **suggestive, not confirmatory**. Genuine confirmation would require an independent dataset, a held-out sample, or a pre-registered replication — not another pass over the same train-set rows.
- As with the rest of this pipeline, any PIU/mental-health association reported below is **preliminary, subset-derived (train-set only), and hypothesis-generating** — never a clinical or diagnostic claim about any individual.

**Loads:** `01_data_preparation.pkl` (raw features only — this notebook does *not* reuse notebook 02's embeddings or labels, by design). Optionally compares against `04_archetype_characterisation.pkl` if available, purely as a cross-check.
**Produces:** `05_three_group_alternative_analysis.pkl` (kept separate from the main-pipeline artifacts).

## Setup & Environment

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
from scipy.stats import kruskal

import umap.umap_ as umap


In [ ]:
!pip -q install umap-learn


### Load only the raw, cleaned features from notebook 01

Deliberately independent of notebook 02: we re-standardise and re-embed from scratch below, rather than reusing the main pipeline's UMAP embedding, so that this check does not simply inherit the main pipeline's choices.

In [ ]:
ARTIFACT_DIR = Path("/kaggle/working/artifacts")

with open(ARTIFACT_DIR / "01_data_preparation.pkl", "rb") as f:
    artifact_01 = pickle.load(f)

model_df = artifact_01["model_df"].copy()
X_imputed = artifact_01["X_imputed"]
CLUSTER_FEATURES = artifact_01["CLUSTER_FEATURES"]

print(f"Loaded model_df {model_df.shape}, X_imputed {X_imputed.shape}")


## Independent re-embedding and forced three-group clustering

Different random seed (`random_state=7` throughout, vs. `42` in the main pipeline) and a wider UMAP neighbourhood (`n_neighbors=30`, vs. `15`), to test whether the archetype-PIU relationship survives a materially different — not just re-run — parameterisation, with `k` fixed at 3 by construction rather than silhouette-selected.

In [ ]:
ALT_RANDOM_STATE = 7
ALT_UMAP_N_NEIGHBORS = 30
ALT_UMAP_MIN_DIST = 0.1
N_GROUPS = 3  # forced, by design of this check - not silhouette-selected

scaler_alt = StandardScaler()
X_scaled_alt = scaler_alt.fit_transform(X_imputed)
X_scaled_alt = pd.DataFrame(X_scaled_alt, columns=CLUSTER_FEATURES, index=X_imputed.index)

alt_reducer = umap.UMAP(
    n_neighbors=ALT_UMAP_N_NEIGHBORS,
    min_dist=ALT_UMAP_MIN_DIST,
    n_components=5,
    random_state=ALT_RANDOM_STATE,
)
alt_embedding = alt_reducer.fit_transform(X_scaled_alt)

alt_kmeans = KMeans(n_clusters=N_GROUPS, n_init=10, random_state=ALT_RANDOM_STATE).fit(alt_embedding)
alt_labels = alt_kmeans.labels_

print(f"Independent run: k={N_GROUPS} (forced), n_neighbors={ALT_UMAP_N_NEIGHBORS}, random_state={ALT_RANDOM_STATE}")
print("Group sizes:")
display(pd.Series(alt_labels).value_counts().sort_index())


## Internal validation of the three-group solution

Same three metrics as the main pipeline (notebook 03), computed here for this forced three-group solution specifically — reported for transparency, not as evidence that three groups is the "right" number (it wasn't chosen by any internal criterion here; it was imposed).

In [ ]:
alt_validation = pd.Series({
    "n_clusters": N_GROUPS,
    "silhouette": silhouette_score(alt_embedding, alt_labels),
    "davies_bouldin": davies_bouldin_score(alt_embedding, alt_labels),
    "calinski_harabasz": calinski_harabasz_score(alt_embedding, alt_labels),
}).round(3)

display(alt_validation.to_frame("three_group_solution"))


## Cross-check against the main pipeline's chosen archetypes

If `04_archetype_characterisation.pkl` exists, we compare this independent three-group solution to the main pipeline's chosen archetype labels on the overlapping rows via Adjusted Rand Index (ARI) and a cross-tabulation — purely descriptive, to see whether the two solutions carve up the same population in a broadly similar way.

In [ ]:
main_artifact_path = ARTIFACT_DIR / "04_archetype_characterisation.pkl"

if main_artifact_path.exists():
    with open(main_artifact_path, "rb") as f:
        artifact_04 = pickle.load(f)
    main_df = artifact_04["model_df"]

    # align on id (rows are the same underlying children, same order from model_df)
    comparison_df = model_df[["id"]].copy()
    comparison_df["alt_archetype"] = alt_labels
    comparison_df = comparison_df.merge(
        main_df[["id", "archetype"]], on="id", how="inner"
    )

    ari_vs_main = adjusted_rand_score(
        comparison_df["archetype"].astype(int), comparison_df["alt_archetype"]
    )
    print(f"ARI between independent 3-group solution and main-pipeline archetypes: {ari_vs_main:.3f}")
    print("(0 = no better than chance agreement, 1 = identical partitions)")

    display(pd.crosstab(comparison_df["archetype"], comparison_df["alt_archetype"]))
else:
    print(
        "04_archetype_characterisation.pkl not found - run notebook 04 first if you want "
        "a cross-check against the main pipeline's chosen archetypes. Continuing with the "
        "independent three-group solution on its own."
    )
    ari_vs_main = None


## Relating the three-group solution to PIU / mental health outcomes

Mirrors the main pipeline's characterisation step (notebook 04, section 10), but for this independent three-group solution only. As before: PIU/mental-health variables were **not** used to form these groups; they are examined only to see whether the direction of any archetype-outcome association is consistent with the main pipeline.

In [ ]:
model_df["alt_archetype"] = alt_labels

outcome_vars = ["PCIAT-PCIAT_Total", "sii", "CGAS-CGAS_Score", "SDS-SDS_Total_T",
                "PreInt_EduHx-computerinternet_hoursday"]

fig, axes = plt.subplots(1, len(outcome_vars), figsize=(22, 4))
for ax, col in zip(axes, outcome_vars):
    sns.boxplot(data=model_df, x="alt_archetype", y=col, ax=ax, palette="Set2")
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.show()

print("Significance of outcome differences across the three groups (Kruskal-Wallis):")
outcome_pvals = {}
for col in outcome_vars:
    groups = [g[col].dropna().values for _, g in model_df.groupby("alt_archetype")]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        continue
    h_stat, p_val = kruskal(*groups)
    outcome_pvals[col] = p_val
    print(f"  {col:42s}  Kruskal-Wallis p={p_val:.4f}")

print("\nPCIAT-PCIAT_Total mean by group (direction check against the main pipeline):")
display(model_df.groupby("alt_archetype")["PCIAT-PCIAT_Total"].mean().round(1))


## Caveats & Call for Independent Confirmation

**Read this section before citing anything above as a finding.**

- This notebook shows that a forced three-group solution, under a materially different embedding/clustering parameterisation, produces groups that can be compared to the main pipeline's archetypes and related (directionally) to PIU/mental-health outcomes. It does **not** by itself establish that three groups is a better, worse, or equally valid description of the population than the main pipeline's chosen k.
- Any directional agreement above (e.g. similar Kruskal-Wallis significance patterns, or a moderate-to-high ARI against the main archetypes) is **corroborating, not confirmatory** — it comes from the same underlying train-set sample, so it cannot rule out that both solutions share the same subset-specific artefacts (e.g. the actigraphy-wear selection bias documented in notebook 01).
- Before this three-group reading is treated as an established result, it needs **independent confirmation**: for example, replication on the competition's held-out test set (once outcomes are available), replication on an entirely different cohort, or a pre-registered analysis plan run by someone other than the original analyst.
- As throughout this pipeline: these are population-level lifestyle patterns from screening-instrument scores, not clinical diagnoses, and must never be used to label or judge any individual child.

## Save artifacts

In [ ]:
artifact = {
    "alt_embedding": alt_embedding,
    "alt_labels": alt_labels,
    "alt_validation": alt_validation,
    "ari_vs_main": ari_vs_main,
    "outcome_pvals": outcome_pvals,
    "ALT_RANDOM_STATE": ALT_RANDOM_STATE,
    "ALT_UMAP_N_NEIGHBORS": ALT_UMAP_N_NEIGHBORS,
    "ALT_UMAP_MIN_DIST": ALT_UMAP_MIN_DIST,
    "N_GROUPS": N_GROUPS,
}

with open(ARTIFACT_DIR / "05_three_group_alternative_analysis.pkl", "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved artifact -> {ARTIFACT_DIR / '05_three_group_alternative_analysis.pkl'}")
print("\nReminder: treat this notebook's results as exploratory pending independent confirmation.")
